# 1b · Mixed-Class Test Answers (human-labelled atoms)

Builds an extra **test** set of answers that mix entailed, contradicted and neutral claims in one paragraph, like a real answer that is mostly right with one wrong or unsupported detail. Every atomic claim gets a **human gold label**.

### Why the atom labels are reliable
When atoms inherit the label of the text they come from, a label is only right if that text held exactly one claim with that label. This notebook makes that true by construction and then checks it by hand:

1. **Labels are planned per sentence, not per answer.** The answer plan fixes each sentence's label and the part of the premise it targets, e.g. `E C N E`. Every entailment and contradiction sentence quotes its evidence.
2. **The real atomiser checks that each sentence is one claim.** Every sentence goes through notebook 2's atomiser. An answer is accepted only if each sentence gives **exactly one atom** that keeps the whole sentence: no clause cut off, negation and hedge kept. Otherwise it is re-sent with the reason. Each atom then belongs to exactly one labelled sentence, so nothing is inherited from the answer.
3. **No answer contradicts itself.**
   - A contradiction sentence must target a part of the premise that no entailment sentence of the same answer restates, so their evidence quotes may not overlap.
   - Only premises with at least two checkable parts are used (direction, number, advice, mechanism, condition). This gives realistic errors such as the right direction with the wrong number, instead of "X raises Y" next to "X lowers Y".
4. **Blind human labels.**
   - Annotators label each atom against the premise only, without the planned label or the answer.
   - With two annotators the notebook reports Cohen's κ, each annotator's agreement with the plan, and the disagreements to adjudicate.
   - Optionally, `AUDIT_MAIN_ATOMS` atoms from the main `atomic_test.csv` are mixed in blind. Their agreement estimates the label noise of the main test set.

### Why test only
- Hand-labelling is feasible for a few hundred atoms, and gold labels matter most where results are measured.
- Training atoms already get their labels from sentences, via the per-sentence labels of notebooks 1 and 2.
- The main 2×2 design stays unchanged; report this set as an extra evaluation of partial support.

### Workflow
1. **Generate** (sections 1-5) → `data/mixed/mixed_answers.csv` and `mixed_atoms.csv`. It is resumable and uses the shared daily budget.
2. **Export** (section 6) → `annotation_sheet_A.csv`, `annotation_sheet_B.csv`. Give one to each annotator, and keep `annotation_key.csv` away from them.
3. **Import** (section 7): put the filled sheets back in `data/mixed/` and run it. If the annotators disagree, fill `final_label` in `adjudication.csv` and run it again.
4. **Outputs:**
   - `data/atomic_test_mixed.csv`: one row per atom, gold `label`;
   - `data/holistic_test_mixed.csv`: one row per answer, label from the gold atoms.

   Both use notebook 2's columns. An answer's label follows notebook 1's rule: any contradiction → contradiction; else any neutral → neutral; else entailment.

**Requirements.** This notebook loads the code of notebook 1 (Gemini) and notebook 2 (scispaCy) into one kernel, so that kernel needs both sets of packages. For example, run `pip install google-genai python-dotenv nest-asyncio` in `thesiskernel`.


In [ ]:
NB1_PATH = "1_AugmentResponses.ipynb"
NB2_CANDIDATES = ["2_Atomisation_patched.ipynb", "2_Atomisation.ipynb"]   # first one that exists is used
ATOMISER_ALLOW_FALLBACK = False   # keep False: gold atoms must come from the same parser as the experiments

SPLIT = "test"
N_ANSWERS = 100                   # one answer per drug pair (and at most one pair per premise)
SEED = 13
# Answer compositions (E = entailment, C = contradiction, N = neutral sentence) and their weights. Most contain all
# three; the E+N ones are neutral answers, so not every answer is a contradiction. The true FActScore of an answer
# (share of entailed atoms) ranges from 0.25 to 0.75.
COMPOSITIONS = {"E E C N": 3, "E C N N": 2, "E E E C N": 2, "E E C C N": 1, "E E E N": 1, "E E N N": 1}
WORDS_PER_SENTENCE = (8, 22)      # answer length limits = these x number of sentences
GATE_MIN_RECALL = 0.8             # share of a sentence's content words its atom must keep
MIXED_PAIRS_PER_REQUEST = 6
MIXED_MAX_ATTEMPTS = 3

AUDIT_MAIN_ATOMS = 100            # atoms of data/atomic_test.csv mixed blind into the sheets (0 = none)
ANNOTATORS = ["A", "B"]           # one sheet each; one annotator works too (no kappa)
OVERWRITE_SHEETS = False          # True rebuilds the annotation sheets and key - this discards filled-in sheets
MIXED_DIR = "data/mixed"


## 0 · Load notebook 1 and notebook 2's atomiser

In [ ]:
import json, os, time
from dataclasses import dataclass, field
from IPython.display import display
from sklearn.metrics import cohen_kappa_score, confusion_matrix


def run_notebook_code(path, namespace, stop_marker, replace=None):
    """Runs the code cells of `path` in `namespace`, in order, up to the first cell that contains `stop_marker`.
    Lines starting with % or ! (pip installs) are skipped."""
    with open(path, encoding="utf-8") as f:
        cells = json.load(f)["cells"]
    for cell in cells:
        if cell["cell_type"] != "code":
            continue
        src = "".join(cell["source"])
        if stop_marker in src:
            return
        src = "\n".join(l for l in src.splitlines() if not l.lstrip().startswith(("%", "!")))
        for old, new in (replace or {}).items():
            src = src.replace(old, new)
        exec(compile(src, path, "exec"), namespace)
    raise ValueError(f"{path}: no cell contains {stop_marker!r}")


# Notebook 1 (prompt rules, plan helpers, validator, request helpers) into this namespace.
run_notebook_code(NB1_PATH, globals(), "await generate_split(")

# Notebook 2's atomiser in its own namespace (the gate and the annotation atoms use exactly this code).
nb2_path = next((p for p in NB2_CANDIDATES if os.path.exists(p)), None)
fda_path = next((p for p in ("../drug_products.csv", "drug_products.csv") if os.path.exists(p)), None)
if nb2_path is None or fda_path is None:
    raise FileNotFoundError(f"notebook 2: {nb2_path}, drug_products.csv: {fda_path} - both are needed for the gate")
ATOMISER = {}
run_notebook_code(nb2_path, ATOMISER, "def load_holistic(",
                  replace={'"../drug_products.csv"': repr(fda_path),
                           "ALLOW_FALLBACK = False": f"ALLOW_FALLBACK = {ATOMISER_ALLOW_FALLBACK}"})
os.makedirs(MIXED_DIR, exist_ok=True)
ANSWERS_PATH, ATOMS_PATH = f"{MIXED_DIR}/mixed_answers.csv", f"{MIXED_DIR}/mixed_atoms.csv"
SKIPPED_PATH = f"{MIXED_DIR}/mixed_skipped.csv"
print(f"\nLoaded {NB1_PATH} and {nb2_path}.")


## 1 · Premise parts and pair selection

A **part** is something in the premise that one sentence can restate or contradict on its own:
- the effect direction ("increased");
- a number ("35%");
- advice ("is recommended");
- a mechanism ("CYP3A4", "clearance");
- a condition ("in patients with renal impairment").

A pair is used only if its premise has at least two parts, so a contradiction can target one part while the entailment sentences restate another. The pair selection is notebook 1's (same pairs as the main test set), with one pair per premise.

In [ ]:
PART_PATTERNS = {
    "number":    re.compile(r"\b\d+(?:[.,]\d+)?\s*(?:%|percent|-?fold|mg|hours?|days?|weeks?)?", re.I),
    "advice":    ADVICE_RE,
    "mechanism": re.compile(r"\b(?:CYP\s?\d\w*|P-glycoprotein|metabolism|metaboli[sz]ed|clearance|absorption|"
                            r"excretion|protein binding|half-life|AUC|Cmax|bioavailability|elimination)\b", re.I),
    "condition": re.compile(r"\b(?:in patients with|in patients receiving|at (?:high|low) doses?|during|if)\b"
                            r"[^,;.]{0,40}", re.I),
}


def premise_parts(premise, d1, d2):
    """[(kind, premise words)] - the parts of the premise one sentence can restate or contradict on its own."""
    body = TAG_RE.sub("", str(premise)).strip()
    masked = body
    for drug in (d1, d2):
        masked = mention_re(drug).sub(lambda m: "#" * len(m.group(0)), masked)
    parts = []
    direction = direction_match(body, (d1, d2))
    if direction:
        parts.append(("direction", direction.group(0)))
    for kind, pattern in PART_PATTERNS.items():
        m = pattern.search(masked)
        if m:
            parts.append((kind, body[m.start():m.end()].strip()))
    return parts


pairs, references = prepare_pairs(SPLIT)
pairs = pairs.assign(parts=[premise_parts(p, a, b) for p, a, b in zip(pairs["premise"], pairs["e1_text"], pairs["e2_text"])])
candidates = pairs[pairs["parts"].map(len) >= 2]
candidates = candidates.sample(frac=1, random_state=SEED).drop_duplicates("premise")
print(f"{len(candidates):,} of {len(pairs):,} {SPLIT} pairs have a premise with >= 2 parts (one pair per premise)")
print(candidates["interaction_type"].value_counts().to_string())
selected = candidates.head(N_ANSWERS)
print(f"\nselected {len(selected)} pairs; parts per premise: {selected['parts'].map(len).value_counts().sort_index().to_dict()}")
print("part kinds:", dict(Counter(kind for parts in selected["parts"] for kind, _ in parts)))


## 2 · Answer plan

For each pair:
1. **Pick a composition** from `COMPOSITIONS`. It must leave at least one part for the entailment sentences.
2. **Shuffle** the order of the sentences.
3. **Assign parts:**
   - Each contradiction sentence gets its own part to contradict. Parts other than the effect direction are preferred, which gives "right effect, wrong detail" errors. If the direction is contradicted, the entailment sentences do not restate the number, since the number belongs to the direction.
   - Entailment sentences restate the remaining parts.
   - The neutral sentences use notebook 1's strategies: a borrowed effect with drug names hidden, or another effect, then an added detail.

In [ ]:
LETTER = {"E": "entailment", "C": "contradiction", "N": "neutral"}
C_STRATEGY = {"direction": "reverse_direction", "advice": "opposite_advice"}       # other parts: conflicting_fact


@dataclass
class MixedPlan:
    labels: List[str]                    # sentence labels, in order
    targets: List[str]                   # what each sentence does, in order
    composition: str
    parts: List[Tuple[str, str]]
    contradiction_strategies: List[str]
    neutral_reference: str
    premise_hedged: bool
    premise_negative: bool
    direction_stem: str
    sentence_labels: Dict[str, List[str]] = field(default_factory=dict)   # notebook 1's validator reads this


def response_label(labels):
    return "contradiction" if "contradiction" in labels else "neutral" if "neutral" in labels else "entailment"


def make_mixed_plan(row, references):
    rng = random.Random(f"{SEED}:mixed:{row['original_id']}")
    body = TAG_RE.sub("", str(row["premise"])).strip()
    d1, d2 = str(row["e1_text"]).strip(), str(row["e2_text"]).strip()
    parts = list(row["parts"])
    direction = direction_match(body, (d1, d2))

    for _ in range(20):
        feasible = [c for c in COMPOSITIONS if c.split().count("C") <= len(parts) - 1]
        composition = rng.choices(feasible, weights=[COMPOSITIONS[c] for c in feasible])[0]
        letters = composition.split()
        rng.shuffle(letters)
        n_c = letters.count("C")
        pool = parts[:]
        rng.shuffle(pool)
        pool.sort(key=lambda p: p[0] == "direction")          # contradict a detail before the direction itself
        c_parts, e_parts = pool[:n_c], pool[n_c:]
        if any(kind == "direction" for kind, _ in c_parts):    # the number belongs to the direction
            e_parts = [p for p in e_parts if p[0] != "number"]
        if e_parts:
            break

    reference = ""
    for _ in range(30):
        candidate = rng.choice(references) if references else ""
        if candidate and candidate != body and not mentions(candidate, d1) and not mentions(candidate, d2):
            reference = mask_drugs(candidate)
            break
    neutral_strategies = ["borrowed_effect" if reference else "other_effect", "added_detail"]

    targets, c_strategies, ci, ei, ni = [], [], 0, 0, 0
    for letter in letters:
        if letter == "C":
            kind, words = c_parts[ci]
            ci += 1
            strategy = C_STRATEGY.get(kind, "conflicting_fact")
            c_strategies.append(strategy)
            targets.append(f"contradiction ({strategy}): conflict with the {kind} part ('{words}') and nothing else")
        elif letter == "E":
            kind, words = e_parts[ei % len(e_parts)]
            again = " in other words" if ei >= len(e_parts) else ""
            ei += 1
            targets.append(f"entailment: restate the {kind} part ('{words}'){again}")
        else:
            strategy = neutral_strategies[ni % 2]
            ni += 1
            if strategy == "added_detail":
                targets.append(f"neutral (added_detail): restate the {e_parts[0][0]} part and add a mechanism, "
                               f"dose or time course the premise does not give")
            elif strategy == "borrowed_effect":
                targets.append("neutral (borrowed_effect): the kind of effect in the reference sentence, "
                               "for Drug 1 and Drug 2 together")
            else:
                targets.append("neutral (other_effect): another effect of the two drugs together that the premise "
                               "does not mention")

    labels = [LETTER[l] for l in letters]
    stem = direction.group(1).lower() if direction and "reverse_direction" in c_strategies else ""
    return MixedPlan(labels=labels, targets=targets, composition=" ".join(letters), parts=parts,
                     contradiction_strategies=c_strategies, neutral_reference=reference,
                     premise_hedged=bool(HEDGE_RE.search(body)), premise_negative=bool(NEGATION_RE.search(body)),
                     direction_stem=stem, sentence_labels={"answer": labels})


items = [{"pair_id": r["original_id"], "premise": r["premise"], "drug1": r["e1_text"], "drug2": r["e2_text"],
          "interaction_type": r.get("interaction_type", "unknown"), "plan": make_mixed_plan(r, references),
          "row": r, "attempts": 0, "feedback": ""} for _, r in selected.iterrows()]
plans = pd.DataFrame([{"pair": it["pair_id"], "composition": it["plan"].composition,
                       "answer label": response_label(it["plan"].labels),
                       "parts": "; ".join(f"{k} '{w}'" for k, w in it["plan"].parts)} for it in items])
print(plans["composition"].value_counts().to_string())
print(plans["answer label"].value_counts().to_string())
display(plans.head(5))


## 3 · Prompt

The rules (correctness, drug names, certainty, negation, banned words, interaction types, strategies, common mistakes) are taken from notebook 1's `SYSTEM_PROMPT`, so both prompts stay the same. Added here: one claim per sentence, and no self-contradiction. The worked example below is checked with the same validator and gate as the model's output.

In [ ]:
def prompt_section(start):
    """One numbered section (or COMMON MISTAKES) of notebook 1's SYSTEM_PROMPT."""
    m = re.search(rf"(?ms)^{re.escape(start)}.*?(?=^\d+\. [A-Z]|^COMMON MISTAKES|^OUTPUT:|\Z)", SYSTEM_PROMPT)
    if not m:
        raise ValueError(f"section {start!r} not found in notebook 1's SYSTEM_PROMPT")
    return m.group(0).strip()


strategies = re.sub(r"The plan names.*?in order\)\.\s*", "", prompt_section("8. STRATEGIES"), flags=re.S)
strategies = (strategies
              .replace("entailment (use the planned strategy for at least one entailment sentence; vary the others)",
                       "entailment (any of these, varied)")
              .replace('contradiction (follow the item\'s "Contradiction target" lines when given)',
                       "contradiction (the answer plan gives each one's strategy and the premise part it targets)"))

MIXED_SYSTEM_PROMPT = f"""
You are a biomedical NLP expert creating a TEST set for a drug-drug interaction (DDI) natural language inference model.

You receive drug-pair items. Each item has a PREMISE sentence, two drug mentions (Drug 1, Drug 2), facts about the
premise, the PARTS of the premise and an ANSWER PLAN. For each item write ONE answer: a short paragraph a clinical
assistant might give about Drug 1 and Drug 2, made of the planned sentences in the planned order. Like a real answer
that is mostly right, it mixes sentences the premise supports (entailment), sentences that conflict with it
(contradiction) and sentences it neither supports nor rules out (neutral). Every sentence carries its own label and
the EVIDENCE for that label.

A. ONE CLAIM PER SENTENCE
Each sentence is ONE short clause that names both drugs and states one effect, fact or piece of advice with a plain
verb (increases, decreases, raises, lowers, prolongs, inhibits, should be avoided, should be reduced, ...). Do not join
a second claim with "and", "which", "so" or "while". Each sentence is checked by a parser that must find exactly one
claim in it, so keep it simple.

B. THE ANSWER MUST NOT CONTRADICT ITSELF
A contradiction sentence conflicts only with the premise part the plan gives it, and no entailment sentence of the
answer restates that part. Quote only the premise words for the part a sentence uses: the evidence of a contradiction
sentence and of an entailment sentence must never overlap.

{prompt_section("1. CORRECTNESS")}

{prompt_section("3. DRUG NAMES")}

{prompt_section("4. CERTAINTY")}

{prompt_section("5. NEGATION")}

{prompt_section("6. BANNED WORDS")}

{prompt_section("7. CORPUS INTERACTION TYPE")}

{strategies}

{prompt_section("COMMON MISTAKES")}

OUTPUT: ONLY a JSON array with one object per item, in the input order. Do NOT repeat the premise:
[
  {{"item_id": "P1", "sentences": [
    {{"label": "entailment", "text": "<sentence>", "evidence": "<exact premise words>"}},
    {{"label": "contradiction", "text": "<sentence>", "evidence": "<exact premise words>", "conflict_type": "<opposite_direction|negates_fact|opposite_advice>"}},
    {{"label": "neutral", "text": "<sentence>", "missing": "<what the premise does not say>"}},
    ... (as many sentences as planned, in the planned order) ...]}},
  ...
]
No markdown, no explanations.
"""

MIXED_EXAMPLE = {
    "premise": "Cimetidine increased the plasma concentrations of procainamide by about 35%.",
    "drug1": "cimetidine", "drug2": "procainamide", "type": "mechanism",
    "parts": [("direction", "increased"), ("number", "35%")],
    "targets": ["entailment: restate the direction part ('increased')",
                "contradiction (conflicting_fact): conflict with the number part ('35%') and nothing else",
                "neutral (added_detail): restate the direction part and add a mechanism, dose or time course the "
                "premise does not give",
                "entailment: restate the direction part ('increased') in other words"],
    "sentences": [
        {"label": "entailment", "text": "Cimetidine increases the plasma concentrations of procainamide.",
         "evidence": "increased the plasma concentrations of procainamide"},
        {"label": "contradiction", "text": "Cimetidine raises procainamide plasma concentrations by roughly 80%.",
         "evidence": "by about 35%", "conflict_type": "negates_fact"},
        {"label": "neutral", "text": "Cimetidine raises procainamide levels by reducing the renal clearance of procainamide.",
         "missing": "the mechanism (renal clearance)"},
        {"label": "entailment", "text": "Procainamide plasma concentrations rise when cimetidine is given.",
         "evidence": "increased the plasma concentrations of procainamide"},
    ],
}


def mixed_item_lines(item_id, drug1, drug2, interaction_type, premise, plan_like, feedback=""):
    hedged, negative = plan_like["hedged"], plan_like["negative"]
    lines = [f"Item ID: {item_id}", f"Drug 1: {drug1}", f"Drug 2: {drug2}", f"Interaction type: {interaction_type}",
             f"Premise: {premise}", f"Premise certainty: {'hedged' if hedged else 'definite'}",
             f"Premise is negative: {'yes' if negative else 'no'}",
             "Premise parts: " + "; ".join(f"{k} ('{w}')" for k, w in plan_like["parts"]),
             f"Answer plan ({len(plan_like['targets'])} sentences, in this order):"]
    lines += [f"  {i}. {t}" for i, t in enumerate(plan_like["targets"], start=1)]
    lines.append(f"Negation words allowed in: {plan_like['negation'] or 'no sentence'}")
    lines.append("Hedge words: entailment and neutral sentences about the hedged claim hedge like the premise"
                 if hedged else "Hedge words: none in any sentence")
    if plan_like.get("reference"):
        lines.append(f"Reference sentence for borrowed_effect (drug names hidden): {plan_like['reference']}")
    if feedback:
        lines.append(f"Your previous answer for this item was rejected: {feedback}. Fix exactly this.")
    return "\n".join(lines)


def plan_like(plan):
    return {"hedged": plan.premise_hedged, "negative": plan.premise_negative, "parts": plan.parts,
            "targets": plan.targets, "negation": ", ".join(negation_allowed(plan)), "reference": plan.neutral_reference}


def build_mixed_prompt(chunk):
    ex = MIXED_EXAMPLE
    example = mixed_item_lines("EXAMPLE", ex["drug1"], ex["drug2"], ex["type"], ex["premise"],
                               {"hedged": False, "negative": False, "parts": ex["parts"], "targets": ex["targets"],
                                "negation": "", "reference": ""})
    blocks = [mixed_item_lines(f"P{i + 1}", it["drug1"].strip(), it["drug2"].strip(), it["interaction_type"],
                               it["premise"].strip(), plan_like(it["plan"]), it.get("feedback", ""))
              for i, it in enumerate(chunk)]
    return ("WORKED EXAMPLE (different drugs; do NOT reuse its wording)\n" + example
            + f"\nOutput sentences: {json.dumps(ex['sentences'])}"
            + f"\n\nNOW WRITE one answer for each of these {len(chunk)} drug pairs:\n\n" + "\n\n".join(blocks)
            + f"\n\nReturn a JSON array with {len(chunk)} objects (one per Item ID).")


def mixed_model_config():
    return model_config().model_copy(update={"system_instruction": MIXED_SYSTEM_PROMPT})


print(f"mixed system prompt: {len(MIXED_SYSTEM_PROMPT):,} characters")


## 4 · Validation and the atomiser gate

An answer is accepted only if all of these hold:
1. **Plan:** the sentence labels match the plan, in order.
2. **Notebook 1's sentence rules:** quotes, both drugs, no other drug, banned words, negation, hedging, no role swap, a neutral must add something, no repeated contradiction or neutral. Length is `WORDS_PER_SENTENCE` × sentences.
3. **No self-contradiction:** contradiction and entailment evidence quotes do not overlap.
4. **Gate:** notebook 2's atomiser gives exactly one atom per sentence. That atom keeps at least `GATE_MIN_RECALL` of the sentence's content words, is not context-trimmed, and keeps the sentence's negation and hedge.

A failed answer is re-sent with the reasons, up to `MIXED_MAX_ATTEMPTS` times. The worked example is checked first.

In [ ]:
def quote_positions(evidence, premise_words):
    quote = normalize(evidence).split()
    for i in range(len(premise_words) - len(quote) + 1):
        if quote and premise_words[i:i + len(quote)] == quote:
            return set(range(i, i + len(quote)))
    return set()


def self_contradiction_problems(sentences, premise):
    premise_words = normalize(premise).split()
    spans = [(k, s["label"], quote_positions(s.get("evidence", ""), premise_words)) for k, s in enumerate(sentences, 1)]
    problems = []
    for k, label, span in spans:
        if label != "contradiction":
            continue
        for j, other, other_span in spans:
            if other == "entailment" and span & other_span:
                problems.append(f"sentence {k} (contradiction) and sentence {j} (entailment) quote the same premise "
                                f"words, so the answer contradicts itself: contradict only the planned part and quote "
                                f"only the words each sentence uses")
    return problems


def atom_gate(sentences, item):
    """Runs notebook 2's atomiser on the answer. Returns (problems, atoms)."""
    claims = [{"text": clean_sentence(s["text"]), "label": s["label"]} for s in sentences]
    answer = pd.DataFrame([{"source_row": 0, "original_id": item["pair_id"], "e1_text": item["drug1"],
                            "e2_text": item["drug2"], "premise": item["premise"],
                            "hypothesis": " ".join(c["text"] for c in claims),
                            "label": response_label([c["label"] for c in claims]),
                            "claims": json.dumps(claims, ensure_ascii=False)}])
    atoms = ATOMISER["atomize_dataframe"](answer, split_name="gate")
    found = atoms[atoms["source_tier"] != "filtered_out"]
    drugs, problems = (item["drug1"], item["drug2"]), []
    for k, claim in enumerate(claims):
        hits = found[found["claim_idx"] == k]
        if len(hits) != 1:
            problems.append(f"sentence {k + 1}: the claim parser finds {len(hits)} claims in it instead of exactly 1 - "
                            f"write one simple clause with both drugs and a plain effect or advice verb")
            continue
        atom = hits.iloc[0]
        words, kept = content_words(claim["text"], drugs), content_words(atom["hypothesis"], drugs)
        recall = len(words & kept) / len(words) if words else 1.0
        if bool(atom["context_trimmed"]) or recall < GATE_MIN_RECALL:
            problems.append(f"sentence {k + 1}: the claim parser keeps only '{atom['hypothesis']}' - drop the extra "
                            f"clause so the whole sentence is one claim")
        elif (bool(NEGATION_RE.search(claim["text"])) != bool(NEGATION_RE.search(atom["hypothesis"]))
              or bool(HEDGE_RE.search(claim["text"])) != bool(HEDGE_RE.search(atom["hypothesis"]))):
            problems.append(f"sentence {k + 1}: the claim parser loses its negation or hedge - put it next to the verb")
    return problems, found


def check_mixed(sentences, item):
    """Returns (accepted sentences or None, hard problems, soft problems, atoms)."""
    plan, n = item["plan"], len(item["plan"].labels)
    if not isinstance(sentences, list) or len(sentences) != n:
        return None, [f"expected {n} sentences"], [], None
    if any(not isinstance(s, dict) or not str(s.get("text", "")).strip() or s.get("label") not in LABELS
           for s in sentences):
        return None, ["malformed sentence"], [], None
    got = [s["label"] for s in sentences]
    if got != plan.labels:
        return None, [f"sentence labels {got} instead of the planned {plan.labels}"], [], None

    global WORD_RANGE
    saved, WORD_RANGE = WORD_RANGE, (WORDS_PER_SENTENCE[0] * n, WORDS_PER_SENTENCE[1] * n)
    try:
        hard = hard_problems({"answer": {"sentences": sentences}}, item)
        soft = soft_problems({"answer": {"sentences": sentences}}, item)
    finally:
        WORD_RANGE = saved
    hard += self_contradiction_problems(sentences, item["premise"])
    atoms = None
    if not hard:
        gate, atoms = atom_gate(sentences, item)
        hard += gate
    return (sentences if not hard else None), hard, soft, atoms


# the worked example must pass everything it teaches
example_plan = MixedPlan(labels=[s["label"] for s in MIXED_EXAMPLE["sentences"]], targets=MIXED_EXAMPLE["targets"],
                         composition="E C N E", parts=MIXED_EXAMPLE["parts"],
                         contradiction_strategies=["conflicting_fact"], neutral_reference="", premise_hedged=False,
                         premise_negative=False, direction_stem="",
                         sentence_labels={"answer": [s["label"] for s in MIXED_EXAMPLE["sentences"]]})
example_item = {"pair_id": "example", "premise": MIXED_EXAMPLE["premise"], "drug1": MIXED_EXAMPLE["drug1"],
                "drug2": MIXED_EXAMPLE["drug2"], "plan": example_plan}
_, hard, soft, _ = check_mixed(json.loads(json.dumps(MIXED_EXAMPLE["sentences"])), example_item)
gate_issues = [p for p in hard if "claim parser" in p]
assert not [p for p in hard if p not in gate_issues], f"the worked example breaks a rule: {hard}"
print("worked example passes the validator" + (f"; atomiser gate: {gate_issues}" if gate_issues else " and the gate"))


## 5 · Generate

Resumable: accepted answers are appended to `mixed_answers.csv` (with their atoms in `mixed_atoms.csv`) after every request. Pairs that fail `MIXED_MAX_ATTEMPTS` times go to `mixed_skipped.csv`. Requests count against the shared daily budget, and a re-run continues where it stopped.

In [ ]:
def call_model(prompt, config_fn):
    """One request, counted against the daily budget. Returns the response, or None when the budget is used up."""
    errors = 0
    while True:
        if not budget.take():
            print(f"Daily request budget used up ({budget.sent}/{DAILY_REQUEST_BUDGET}).")
            return None
        try:
            return client.models.generate_content(model=MODEL_NAME, contents=prompt, config=config_fn())
        except Exception as e:
            if _thinking["enabled"] and "thinking" in str(e).lower():
                budget.refund()
                _thinking["enabled"] = False
                continue
            is_daily, retry_delay = classify_rate_limit(e)
            if is_daily:
                budget.exhaust()
                print("Daily quota reached (re-run after the reset, 14:00 WIB).")
                return None
            if retry_delay is not None:
                budget.refund()
                time.sleep(retry_delay + 1)
                continue
            errors += 1
            if errors >= TRANSPORT_RETRIES:
                raise
            time.sleep(min(2.0 ** errors, 60.0))


def parse_mixed(text, chunk):
    try:
        data = json.loads((text or "").strip())
    except json.JSONDecodeError:
        data = None
    if not isinstance(data, list):
        return {it["pair_id"]: (None, ["the response was not a JSON array"], [], None) for it in chunk}
    by_item_id, out = {f"P{i + 1}": it for i, it in enumerate(chunk)}, {}
    for entry in data:
        item = by_item_id.get(str(entry.get("item_id", "")).strip()) if isinstance(entry, dict) else None
        if item is not None and item["pair_id"] not in out:
            out[item["pair_id"]] = check_mixed(entry.get("sentences"), item)
    for it in chunk:
        out.setdefault(it["pair_id"], (None, ["the item was missing from the response"], [], None))
    return out


def answer_row(item, sentences, soft):
    claims = []
    for s in sentences:
        claim = {"text": clean_sentence(s["text"]), "label": s["label"]}
        for key in ("evidence", "conflict_type", "missing"):
            if s.get(key):
                claim[key] = s[key]
        claims.append(claim)
    r, plan = item["row"], item["plan"]
    return {"answer_id": item["pair_id"], "original_id": r["original_id"], "e1_text": r["e1_text"],
            "e2_text": r["e2_text"], "premise": r["premise"], "hypothesis": " ".join(c["text"] for c in claims),
            "label": response_label(plan.labels), "claims": json.dumps(claims, ensure_ascii=False),
            "claim_labels": " ".join(l[0].upper() for l in plan.labels), "composition": plan.composition,
            "parts": json.dumps(plan.parts), "targets": json.dumps(plan.targets),
            "source_interaction_type": r.get("interaction_type", "unknown"), "style_violations": "; ".join(soft),
            "gen_version": "mixed"}


def append_csv(df, path):
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)


done = set(pd.read_csv(ANSWERS_PATH)["answer_id"]) if os.path.exists(ANSWERS_PATH) else set()
done |= set(pd.read_csv(SKIPPED_PATH)["answer_id"]) if os.path.exists(SKIPPED_PATH) else set()
queue = [it for it in items if it["pair_id"] not in done]
print(f"{len(done)} pairs already done or skipped, {len(queue)} to generate")
reasons = Counter()
while queue:
    chunk, queue = queue[:MIXED_PAIRS_PER_REQUEST], queue[MIXED_PAIRS_PER_REQUEST:]
    response = call_model(build_mixed_prompt(chunk), mixed_model_config)
    if response is None:
        print("stopped; re-run this cell after the reset to continue")
        break
    outcome = parse_mixed(response.text, chunk)
    for item in chunk:
        sentences, hard, soft, atoms = outcome[item["pair_id"]]
        if sentences is not None:
            append_csv(pd.DataFrame([answer_row(item, sentences, soft)]), ANSWERS_PATH)
            append_csv(atoms.assign(answer_id=item["pair_id"]), ATOMS_PATH)
            continue
        reasons.update(reason_key(p) for p in hard)
        if item["attempts"] + 1 >= MIXED_MAX_ATTEMPTS:
            append_csv(pd.DataFrame([{"answer_id": item["pair_id"], "reason": "; ".join(hard[:3])}]), SKIPPED_PATH)
        else:
            queue.append({**item, "attempts": item["attempts"] + 1, "feedback": "; ".join(hard[:3])})
    time.sleep(60.0 / REQUESTS_PER_MINUTE)

answers = pd.read_csv(ANSWERS_PATH) if os.path.exists(ANSWERS_PATH) else pd.DataFrame()
print(f"\naccepted answers: {len(answers)}  (requests used today: {budget.sent}/{DAILY_REQUEST_BUDGET})")
if len(answers):
    print(answers["composition"].value_counts().to_string())
    print(answers["label"].value_counts().to_string())
for reason, n in reasons.most_common():
    print(f"  rejected {n:3d}x  {reason}")


## 6 · Export the annotation sheets

One sheet per annotator, each in its own random order. A sheet shows only the premise, the two drugs and the atom: no planned label, no answer, no source. The audit atoms from the main test set are mixed in the same way.

**Annotation guideline.** Read the premise, then the atom. Write one label in `label`:
- **entailment (e):** the atom must be true if the premise is true.
- **contradiction (c):** the atom cannot be true if the premise is true. It is about the same drugs and the same measure, and the two are incompatible.
- **neutral (n):** the premise neither confirms nor rules out the atom. Added details count here: a mechanism, dose or population the premise does not mention.

Hedges matter: "may increase" is not the same claim as "increases". Swapping which drug affects which is neutral, not contradiction. Use `unsure` (y) for hard cases, and `notes` for anything else. Do not look anything up: judge against the premise only.

In [ ]:
KEY_PATH = f"{MIXED_DIR}/annotation_key.csv"


def export_sheets():
    answers = pd.read_csv(ANSWERS_PATH)
    mixed_atoms = pd.read_csv(ATOMS_PATH)
    mixed_atoms = mixed_atoms[mixed_atoms["answer_id"].isin(answers["answer_id"])]
    key = pd.DataFrame({
        "source": "mixed", "answer_id": mixed_atoms["answer_id"].values, "sentence": mixed_atoms["claim_idx"].values + 1,
        "planned_label": mixed_atoms["label"].values, "premise": mixed_atoms["premise"].values,
        "drug_1": mixed_atoms["Entity1"].values, "drug_2": mixed_atoms["Entity2"].values,
        "atom": mixed_atoms["hypothesis"].values})

    main_path = "data/atomic_test.csv"
    if AUDIT_MAIN_ATOMS and os.path.exists(main_path):
        main = pd.read_csv(main_path)
        main = main.sample(n=min(AUDIT_MAIN_ATOMS, len(main)), random_state=SEED)
        key = pd.concat([key, pd.DataFrame({
            "source": "main_test_audit", "answer_id": main["source_row"].astype(str).values,
            "sentence": main["claim_idx"].values + 1, "planned_label": main["label"].values,
            "premise": main["premise"].values, "drug_1": main["Entity1"].values, "drug_2": main["Entity2"].values,
            "atom": main["hypothesis"].values})], ignore_index=True)
    elif AUDIT_MAIN_ATOMS:
        print(f"{main_path} not found: no audit atoms")

    key = key.sample(frac=1, random_state=SEED).reset_index(drop=True)
    key.insert(0, "atom_id", [f"X{i:04d}" for i in range(1, len(key) + 1)])
    key.to_csv(KEY_PATH, index=False)
    for k, annotator in enumerate(ANNOTATORS):
        sheet = key[["atom_id", "premise", "drug_1", "drug_2", "atom"]].sample(frac=1, random_state=SEED + 1 + k)
        sheet = sheet.assign(label="", unsure="", notes="")
        sheet.insert(0, "item", range(1, len(sheet) + 1))
        sheet.to_csv(f"{MIXED_DIR}/annotation_sheet_{annotator}.csv", index=False)
    print(f"{len(key)} atoms ({(key['source'] == 'mixed').sum()} mixed, {(key['source'] != 'mixed').sum()} audit) -> "
          + ", ".join(f"{MIXED_DIR}/annotation_sheet_{a}.csv" for a in ANNOTATORS)
          + f"; keep {MIXED_DIR}/annotation_key.csv away from the annotators")


if os.path.exists(KEY_PATH) and not OVERWRITE_SHEETS:
    print(f"{KEY_PATH} already exists: the sheets were exported before and may be filled in, so they are left alone. "
          f"Set OVERWRITE_SHEETS = True to rebuild them (this discards their labels).")
else:
    export_sheets()


## 7 · Import the annotations → gold labels

1. **Read** the filled sheets. Labels may be written as `e`, `c` or `n`, or in full.
2. **Report:**
   - how many atoms each annotator labelled;
   - Cohen's κ between the first two annotators;
   - each annotator's agreement with the planned labels, overall and per source. For the audit atoms this is the label-noise estimate of the main test set.
3. **Gold:** an atom's gold label is the label all annotators agree on. Disagreements go to `adjudication.csv`: fill `final_label` there and run this cell again.
4. **Outputs:** once every mixed atom has a gold label, this writes:
   - `data/atomic_test_mixed.csv`: notebook 2's atom columns, with gold `label` and `planned_label`;
   - `data/holistic_test_mixed.csv`: one row per answer with `source_row`, the gold `label` from its atoms, `f_true` (share of entailed atoms) and `covered`.

In [ ]:
ALIASES = {"e": "entailment", "ent": "entailment", "entail": "entailment", "entailment": "entailment",
           "c": "contradiction", "con": "contradiction", "contra": "contradiction", "contradiction": "contradiction",
           "n": "neutral", "neu": "neutral", "neutral": "neutral"}


def read_sheet(annotator):
    path = f"{MIXED_DIR}/annotation_sheet_{annotator}.csv"
    sheet = pd.read_csv(path, dtype=str).fillna("")
    labels = sheet.set_index("atom_id")["label"].str.strip().str.lower().map(ALIASES)
    bad = sheet.loc[sheet["label"].str.strip().ne("") & labels.reindex(sheet["atom_id"]).isna().values, "label"]
    if len(bad):
        print(f"[{annotator}] unreadable labels ignored: {sorted(set(bad))}")
    return labels



def import_annotations():
    key = pd.read_csv(KEY_PATH)
    labels = pd.DataFrame({a: read_sheet(a) for a in ANNOTATORS}).reindex(key["atom_id"])
    merged = key.set_index("atom_id").join(labels)

    for a in ANNOTATORS:
        done = merged[a].notna()
        print(f"annotator {a}: {done.sum()}/{len(merged)} atoms labelled")
        for source, part in merged[done].groupby("source"):
            agree = (part[a] == part["planned_label"]).mean()
            kappa = cohen_kappa_score(part[a], part["planned_label"]) if part[a].nunique() > 1 else float("nan")
            print(f"   {source:16s} agreement with the planned label {agree:.1%} (kappa {kappa:.3f}, n={len(part)})")
    if len(ANNOTATORS) >= 2:
        a, b = ANNOTATORS[:2]
        both = merged[a].notna() & merged[b].notna()
        if both.any():
            print(f"\nCohen's kappa {a} vs {b}: {cohen_kappa_score(merged.loc[both, a], merged.loc[both, b]):.3f} "
                  f"(raw agreement {(merged.loc[both, a] == merged.loc[both, b]).mean():.1%}, n={both.sum()})")
    first = merged[merged[ANNOTATORS[0]].notna()]
    if len(first):
        print("\nplanned (rows) x annotator", ANNOTATORS[0], "(columns), all labelled atoms:")
        display(pd.crosstab(first["planned_label"], first[ANNOTATORS[0]]))

    # gold = unanimous label; disagreements are adjudicated
    votes = merged[ANNOTATORS]
    complete = votes.notna().all(axis=1)
    unanimous = complete & votes.nunique(axis=1).eq(1)
    merged["gold"] = votes.iloc[:, 0].where(unanimous)
    adj_path = f"{MIXED_DIR}/adjudication.csv"
    disagree = merged[complete & ~unanimous]
    if os.path.exists(adj_path):
        adjudicated = pd.read_csv(adj_path, dtype=str).fillna("").set_index("atom_id")["final_label"]
        adjudicated = adjudicated.str.strip().str.lower().map(ALIASES).dropna()
        merged.loc[adjudicated.index.intersection(merged.index), "gold"] = adjudicated
    if len(disagree):
        sheet = disagree[["premise", "atom"] + ANNOTATORS].copy()
        previous = (pd.read_csv(adj_path, dtype=str).fillna("").set_index("atom_id")["final_label"]
                    if os.path.exists(adj_path) else pd.Series(dtype=str))
        sheet["final_label"] = previous.reindex(sheet.index).fillna("")
        sheet.reset_index().to_csv(adj_path, index=False)
        open_cases = int((sheet["final_label"].str.strip() == "").sum())
        print(f"\n{len(disagree)} disagreements in {adj_path}"
              + (f": {open_cases} still need a final_label (fill it, then re-run)" if open_cases else ", all adjudicated"))

    mixed = merged[merged["source"] == "mixed"]
    missing = mixed["gold"].isna().sum()
    audit = merged[(merged["source"] != "mixed") & merged["gold"].notna()]
    if len(audit):
        print(f"\nmain test set audit: {(audit['gold'] == audit['planned_label']).mean():.1%} of {len(audit)} atoms keep "
              f"their pipeline label under the gold label")
    if missing:
        print(f"\n{missing} mixed atoms have no gold label yet (unlabelled or not adjudicated): outputs not written")
    else:
        gold = mixed.reset_index()[["answer_id", "sentence", "gold"]]
        atoms = pd.read_csv(ATOMS_PATH)
        atoms = atoms.assign(sentence=atoms["claim_idx"] + 1).merge(gold, on=["answer_id", "sentence"], how="inner")
        answers = pd.read_csv(ANSWERS_PATH)
        answers = answers[answers["answer_id"].isin(atoms["answer_id"])].reset_index(drop=True)
        answers.insert(0, "source_row", range(len(answers)))
        gold_by_answer = atoms.groupby("answer_id")["gold"].apply(list)
        answers["planned_label"] = answers["label"]
        answers["label"] = answers["answer_id"].map(gold_by_answer).map(response_label)
        answers["n_atoms"] = answers["answer_id"].map(gold_by_answer.map(len))
        answers["f_true"] = answers["answer_id"].map(gold_by_answer.map(lambda g: g.count("entailment") / len(g)))
        answers["covered"] = True

        row_of = answers.set_index("answer_id")["source_row"]
        atoms = atoms.assign(source_row=atoms["answer_id"].map(row_of), planned_label=atoms["label"],
                             label=atoms["gold"], response_label=atoms["answer_id"].map(answers.set_index("answer_id")["label"]),
                             label_source="annotated").drop(columns=["gold"])
        atoms.to_csv("data/atomic_test_mixed.csv", index=False)
        answers.to_csv("data/holistic_test_mixed.csv", index=False)
        print(f"\nwrote data/atomic_test_mixed.csv ({len(atoms)} atoms) and data/holistic_test_mixed.csv ({len(answers)} answers)")
        print("gold answer labels:", answers["label"].value_counts().to_dict(),
              f"| answers whose label changed from the plan: {(answers['label'] != answers['planned_label']).sum()}")
        print("gold atom labels:", atoms["label"].value_counts().to_dict())


if os.path.exists(KEY_PATH):
    import_annotations()
else:
    print(f"{KEY_PATH} not found: export the annotation sheets first (section 6)")
